In [ ]:
# ======================================================
# Notebook: Drug Combination Optimisation (Neural Network)
# Maximise (- side effects)
# ======================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,3)
y_raw = np.load("/mnt/data/initial_outputs.npy") # (N,)

# Transform objective
y = -y_raw
y = (y - y.mean()) / (y.std() + 1e-8)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y.reshape(-1,1), dtype=torch.float32)

# Neural network surrogate
model = nn.Sequential(
    nn.Linear(3, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 1)
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train
model.train()
for _ in range(600):
    optimizer.zero_grad()
    pred = model(X_t)
    loss = criterion(pred, y_t)
    loss.backward()
    optimizer.step()

# Candidate grid
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(3)]
grid = [np.linspace(b[0], b[1], 20) for b in bounds]
X_grid = np.array(np.meshgrid(*grid)).T.reshape(-1,3)
X_grid_t = torch.tensor(X_grid, dtype=torch.float32)

# MC Dropout uncertainty
model.train()
samples = []

with torch.no_grad():
    for _ in range(30):
        samples.append(model(X_grid_t).numpy())

samples = np.stack(samples)
mean_pred = samples.mean(axis=0).flatten()
uncertainty = samples.std(axis=0).flatten()

# Acquisition
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,3)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,3) compound combinations:")
print(next_points)